In [1]:
import random
print("라이브러리 로드 완료")

라이브러리 로드 완료


In [2]:
# ── 설정 ──
INPUT_FILE  = 'address_raw.txt'    # 원본 도로명주소 파일명
OUTPUT_FILE = 'addresses_data.sql' # 출력 파일명
USER_COUNT  = 100                  # users_data.sql과 동일하게 맞춰주세요

In [3]:
def parse_address_line(line: str) -> dict | None:
    parts = line.strip().split('|')
    if len(parts) < 13:
        return None

    sido         = parts[2].strip()
    sigungu      = parts[3].strip()
    road_name    = parts[10].strip()
    building_no  = parts[12].strip()
    building_sub = parts[8].strip()
    building_nm  = parts[22].strip() if len(parts) > 22 else ""

    if not sido or not road_name or not building_no or building_no == '0':
        return None

    if building_sub and building_sub != '0':
        road_full = f"{sido} {sigungu} {road_name} {building_no}-{building_sub}"
    else:
        road_full = f"{sido} {sigungu} {road_name} {building_no}"

    if building_nm:
        detail = f"{building_nm} {random.randint(1,20)}층 {random.randint(101,999)}호"
    else:
        detail = f"{random.randint(1,20)}동 {random.randint(101,999)}호"

    return {
        'road_name_address': road_full,
        'address_detail':    detail,
    }

print("함수 정의 완료")

함수 정의 완료


In [4]:
addresses = []

with open(INPUT_FILE, 'r', encoding='cp949') as f:
    for line in f:
        if not line.strip():
            continue
        parsed = parse_address_line(line)
        if parsed:
            addresses.append(parsed)

print(f"파싱된 주소 {len(addresses)}개")

파싱된 주소 524608개


In [5]:
rows = []
addr_id = 1

for user_id in range(1, USER_COUNT + 1):
    count       = random.randint(1, 3)
    has_default = False
    for j in range(count):
        addr = random.choice(addresses)
        is_default = not has_default and (j == 0 or random.random() < 0.3)
        if is_default:
            has_default = True
        road   = addr['road_name_address'].replace("'", "''")
        detail = addr['address_detail'].replace("'", "''")
        rows.append(
            f"  ({addr_id}, {str(is_default).upper()}, '{road}', '{detail}', {user_id})"
        )
        addr_id += 1

sql = (
    "INSERT INTO address (address_id, is_default, road_name_address, address_detail, user_id) VALUES\n"
    + ',\n'.join(rows) + ';'
)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(sql)

print(f"완료: {OUTPUT_FILE} 생성됨 (배송지 {addr_id-1}개, 유저 {USER_COUNT}명)")

완료: addresses_data.sql 생성됨 (배송지 179개, 유저 100명)


In [6]:
print("\n샘플 3개:")
for r in rows[:3]:
    print(" ", r)


샘플 3개:
    (1, TRUE, '서울특별시 종로구 자하문로24길 49-2', '해공신익희가옥 17층 185호', 1)
    (2, TRUE, '서울특별시 강서구 초록마을로10길 6-44', '6동 949호', 2)
    (3, TRUE, '서울특별시 동대문구 제기로15길 12-124', '9동 990호', 3)
